### Lean Run Guide
This notebook intentionally keeps the training loop simple and transparent.
Goals:
- maximize transferable signal first (spectral + light geo context),
- keep target-wise model choice,
- avoid heavy sweep complexity until baseline behavior is stable.


## Environment and MLflow
We keep imports explicit and logging optional.


In [16]:
import os
import json
import time
import hashlib
import joblib
import mlflow
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.base import clone
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import HistGradientBoostingRegressor


## Global config
This section sets targets, split strategy, and realistic local ceilings.


In [17]:
TARGET_COLS = [
    'Total Alkalinity',
    'Electrical Conductance',
    'Dissolved Reactive Phosphorus'
]
# Realistic local-fold expectations under spatial shift.
REALISTIC_LOCAL_CEILING = {
    'Total Alkalinity': (0.30, 0.45),
    'Electrical Conductance': (0.25, 0.40),
    'Dissolved Reactive Phosphorus': (0.05, 0.15),
}
SPLIT_STRATEGY = 'SimpleSpatialGroupKFold+PseudoHoldout'
GROUP_DEFINITION_VERSION = 'kmeans_latlon_simple_v1'
PIPELINE_VERSION = '03d_simple_baseline_v1'
PREPROCESS_VERSION = 'median_robustscaler_numeric'
SPATIAL_N_CLUSTERS = 12
CV_N_SPLITS = 5
HOLDOUT_MARGIN_DEG = 0.20
HOLDOUT_MIN_GROUPS = 2
ARTIFACT_DIR = '../models/final_deadline_simple'
os.makedirs(ARTIFACT_DIR, exist_ok=True)
print('Config loaded.')
print('Realistic local ceilings:')
for t, (lo, hi) in REALISTIC_LOCAL_CEILING.items():
    print(f'- {t}: ~{lo:.2f} to {hi:.2f}')


Config loaded.
Realistic local ceilings:
- Total Alkalinity: ~0.30 to 0.45
- Electrical Conductance: ~0.25 to 0.40
- Dissolved Reactive Phosphorus: ~0.05 to 0.15


## Data loading
Use raw master sentinel data directly (train/test have matching shape).


In [18]:
TRAIN_PATH = '../data/interim/master_train_sentinel.parquet'
VALID_PATH = '../data/interim/master_test_sentinel.parquet'
CONTRACT_TXT_PATH = '../data/interim/feature_contract_master_sentinel.txt'

for label, path in [('TRAIN_PATH', TRAIN_PATH), ('VALID_PATH', VALID_PATH), ('CONTRACT_TXT_PATH', CONTRACT_TXT_PATH)]:
    if not os.path.exists(path):
        raise RuntimeError(f'{label} not found: {path}')
    print(f'{label} selected: {path}')

df = pd.read_parquet(TRAIN_PATH).copy()
df_test = pd.read_parquet(VALID_PATH).copy()
required_train = ['Latitude', 'Longitude', 'Sample Date'] + TARGET_COLS
missing_train = [c for c in required_train if c not in df.columns]
if missing_train:
    raise RuntimeError(f'Missing training columns: {missing_train}')
required_test_geo = ['Latitude', 'Longitude', 'Sample Date']
missing_test = [c for c in required_test_geo if c not in df_test.columns]
if missing_test:
    raise RuntimeError(f'Missing test geo columns: {missing_test}')
with open(CONTRACT_TXT_PATH, 'r', encoding='utf-8') as f:
    contract_cols = [line.strip() for line in f.readlines() if line.strip()]
missing_contract_train = [c for c in contract_cols if c not in df.columns]
missing_contract_test = [c for c in contract_cols if c not in df_test.columns]
if missing_contract_train:
    raise RuntimeError(f'Contract missing in train ({len(missing_contract_train)}): {missing_contract_train[:12]}')
if missing_contract_test:
    raise RuntimeError(f'Contract missing in test ({len(missing_contract_test)}): {missing_contract_test[:12]}')
def add_spatial_groups(data, n_clusters=SPATIAL_N_CLUSTERS):
    out = data.copy()
    n_clusters = min(max(4, int(n_clusters)), len(out))
    km = KMeans(n_clusters=n_clusters, random_state=42, n_init=20)
    out['spatial_group'] = km.fit_predict(out[['Latitude', 'Longitude']].astype(float)).astype(str)
    return out
def select_holdout_groups_simple(train_df, valid_df, min_groups=HOLDOUT_MIN_GROUPS, margin_deg=HOLDOUT_MARGIN_DEG):
    gdf = train_df.groupby('spatial_group', as_index=False).agg(
        Latitude=('Latitude', 'mean'),
        Longitude=('Longitude', 'mean'),
        n=('spatial_group', 'size')
    )
    lat_min = float(valid_df['Latitude'].min()) - margin_deg
    lat_max = float(valid_df['Latitude'].max()) + margin_deg
    lon_min = float(valid_df['Longitude'].min()) - margin_deg
    lon_max = float(valid_df['Longitude'].max()) + margin_deg
    in_bbox = (
        gdf['Latitude'].between(lat_min, lat_max) &
        gdf['Longitude'].between(lon_min, lon_max)
    )
    valid_center_lat = float(valid_df['Latitude'].mean())
    valid_center_lon = float(valid_df['Longitude'].mean())
    gdf['center_dist'] = np.sqrt(
        (gdf['Latitude'].astype(float) - valid_center_lat) ** 2 +
        (gdf['Longitude'].astype(float) - valid_center_lon) ** 2
    )
    picked = gdf.loc[in_bbox].sort_values(['center_dist', 'n'], ascending=[True, False])['spatial_group'].astype(str).tolist()
    if len(picked) < int(min_groups):
        fallback = gdf.sort_values(['center_dist', 'n'], ascending=[True, False])['spatial_group'].astype(str).tolist()
        for g in fallback:
            if g not in picked:
                picked.append(g)
            if len(picked) >= int(min_groups):
                break
    return picked
df['Sample Date'] = pd.to_datetime(df['Sample Date'], errors='coerce')
df_test['Sample Date'] = pd.to_datetime(df_test['Sample Date'], errors='coerce')
df = add_spatial_groups(df, n_clusters=SPATIAL_N_CLUSTERS)
holdout_groups = select_holdout_groups_simple(df, df_test[['Latitude', 'Longitude']], min_groups=HOLDOUT_MIN_GROUPS)
holdout_set = set(holdout_groups)
df['is_pseudo_valid'] = df['spatial_group'].astype(str).isin(holdout_set)
print('Data ready.')
print('n_rows:', len(df))
print('n_spatial_groups:', df['spatial_group'].nunique())
print('holdout_groups:', sorted(list(holdout_set)))
print('holdout_rows:', int(df['is_pseudo_valid'].sum()), '| holdout_frac:', round(float(df['is_pseudo_valid'].mean()), 4))


TRAIN_PATH selected: ../data/interim/master_train_sentinel.parquet
VALID_PATH selected: ../data/interim/master_test_sentinel.parquet
CONTRACT_TXT_PATH selected: ../data/interim/feature_contract_master_sentinel.txt
Data ready.
n_rows: 9319
n_spatial_groups: 12
holdout_groups: ['5', '9']
holdout_rows: 1097 | holdout_frac: 0.1177


## Feature engineering
Keep only low-risk temporal and ratio features.


In [19]:
def engineer_features(df_local):
    out = df_local.copy()

    if 'Sample Date' in out.columns:
        dts = pd.to_datetime(out['Sample Date'], errors='coerce')
        month = dts.dt.month.fillna(1).astype(int)
        out['month_sin'] = np.sin(2 * np.pi * month / 12.0)
        out['month_cos'] = np.cos(2 * np.pi * month / 12.0)

    if 'terra_ppt' in out.columns and 'terra_pet' in out.columns:
        out['aridity_idx'] = out['terra_ppt'].astype(float) / (out['terra_pet'].astype(float) + 1e-6)

    if 'worldpop_sum_1km' in out.columns and 'worldpop_sum_5km' in out.columns:
        out['pop_local_share_1km'] = out['worldpop_sum_1km'].astype(float) / (out['worldpop_sum_5km'].astype(float) + 1e-6)

    # TA-oriented proxies: geochemistry, residence time, soil weathering, rainfall dilution.
    if 'soil_silt_mean_0_5cm' in out.columns and 'soil_clay_mean_0_5cm' in out.columns and 'soil_sand_mean_0_5cm' in out.columns:
        out['soil_fines_0_5cm'] = out['soil_silt_mean_0_5cm'].astype(float) + out['soil_clay_mean_0_5cm'].astype(float)
        out['soil_texture_balance'] = out['soil_fines_0_5cm'] / (out['soil_sand_mean_0_5cm'].astype(float) + 1e-6)

    if 'river_avg_discharge_cms' in out.columns and 'basin_upstream_area_km2' in out.columns:
        out['residence_time_proxy'] = out['basin_upstream_area_km2'].astype(float) / (out['river_avg_discharge_cms'].astype(float) + 1e-6)

    if 'dem_slope_1km' in out.columns:
        out['flatness_proxy'] = 1.0 / (out['dem_slope_1km'].astype(float) + 0.1)

    if 'basin_slope_deg' in out.columns:
        out['basin_flatness_proxy'] = 1.0 / (out['basin_slope_deg'].astype(float) + 0.1)

    if 'terra_ppt' in out.columns and 'residence_time_proxy' in out.columns:
        out['rain_dilution_proxy'] = out['terra_ppt'].astype(float) / (out['residence_time_proxy'].astype(float) + 1e-6)

    if 'weather_precip_7d_sum' in out.columns and 'river_avg_discharge_cms' in out.columns:
        out['event_runoff_proxy'] = out['weather_precip_7d_sum'].astype(float) * out['river_avg_discharge_cms'].astype(float)

    if 'soil_cec_mean_0_5cm' in out.columns and 'soil_phh2o_mean_0_5cm' in out.columns and 'soil_sand_mean_0_5cm' in out.columns:
        out['weathering_capacity_proxy'] = (
            out['soil_cec_mean_0_5cm'].astype(float) * out['soil_phh2o_mean_0_5cm'].astype(float)
        ) / (out['soil_sand_mean_0_5cm'].astype(float) + 1e-6)

    if 'residence_time_proxy' in out.columns and 'soil_texture_balance' in out.columns:
        out['contact_weathering_proxy'] = out['residence_time_proxy'].astype(float) * out['soil_texture_balance'].astype(float)

    return out

df = engineer_features(df)
df_test = engineer_features(df_test)
print('Feature engineering complete.')


Feature engineering complete.


## Frozen feature sets
Simple, physics-first feature bundles.


In [20]:
RESERVED_COLS = set(TARGET_COLS + ['spatial_group', 'is_pseudo_valid', 'Sample Date', 'Sample_Date'])

SPECTRAL_CANDIDATES = [
    'nir', 'green', 'swir16', 'swir22', 'NDMI', 'MNDWI',
    'sentinel_blue', 'sentinel_green', 'sentinel_red', 'sentinel_nir',
    'sentinel_swir16', 'sentinel_swir22',
    'sentinel_NDMI', 'sentinel_MNDWI', 'sentinel_NDTI', 'sentinel_CDOM',
    'sentinel_NDVI', 'sentinel_EVI', 'sentinel_BSI',
]

LITE_CONTEXT_CANDIDATES = [
    'Latitude', 'Longitude',
    'dem_elev_mean_1km', 'dem_slope_1km',
    'weather_precip_7d_sum', 'weather_temp_7d_max', 'weather_wind_7d_mean',
    'soil_phh2o_mean_0_5cm', 'soil_clay_mean_0_5cm', 'soil_sand_mean_0_5cm',
    'worldpop_mean_1km', 'worldpop_sum_1km',
    'basin_slope_deg', 'river_avg_discharge_cms', 'distance_to_river_m',
    'month_sin', 'month_cos', 'aridity_idx', 'pop_local_share_1km'
]

# TA-specific combinations (avoid generic MIXED_LITE for TA)
TA_HYDRO_GEO_CANDIDATES = [
    'Latitude', 'Longitude',
    'swir22', 'NDMI', 'MNDWI', 'sentinel_CDOM', 'sentinel_NDTI',
    'dem_elev_mean_1km', 'dem_slope_1km', 'basin_slope_deg',
    'basin_upstream_area_km2', 'river_avg_discharge_cms', 'river_upstream_area_km2',
    'river_order', 'distance_to_river_m', 'river_width_m',
    'terra_ppt', 'weather_precip_7d_sum',
    'soil_phh2o_mean_0_5cm', 'soil_clay_mean_0_5cm', 'soil_sand_mean_0_5cm',
    'soil_silt_mean_0_5cm', 'soil_organic_carbon_mean_0_5cm', 'soil_cec_mean_0_5cm',
    'sanlc2022_pct_Macro_Natural_1km', 'sanlc2022_pct_Macro_Mining_1km',
    'sanlc2022_pct_Macro_Agriculture_1km', 'sanlc2022_pct_Macro_Urban_1km',
    'month_sin', 'month_cos', 'aridity_idx',
    'residence_time_proxy', 'flatness_proxy', 'basin_flatness_proxy',
    'rain_dilution_proxy', 'weathering_capacity_proxy', 'contact_weathering_proxy',
]

TA_RESIDENCE_SOIL_CANDIDATES = [
    'Latitude', 'Longitude',
    'dem_slope_1km', 'basin_slope_deg', 'basin_upstream_area_km2',
    'river_avg_discharge_cms', 'river_upstream_area_km2',
    'terra_ppt', 'weather_precip_7d_sum',
    'soil_phh2o_mean_0_5cm', 'soil_clay_mean_0_5cm', 'soil_sand_mean_0_5cm',
    'soil_silt_mean_0_5cm', 'soil_organic_carbon_mean_0_5cm', 'soil_cec_mean_0_5cm',
    'soil_fines_0_5cm', 'soil_texture_balance',
    'residence_time_proxy', 'flatness_proxy', 'basin_flatness_proxy',
    'rain_dilution_proxy', 'weathering_capacity_proxy', 'event_runoff_proxy',
    'contact_weathering_proxy'
]

spectral_core = [c for c in SPECTRAL_CANDIDATES if c in df.columns]
spectral_plus_coords = list(dict.fromkeys(spectral_core + [c for c in ['Latitude', 'Longitude'] if c in df.columns]))
mixed_lite = list(dict.fromkeys(spectral_core + [c for c in LITE_CONTEXT_CANDIDATES if c in df.columns]))
ta_hydro_geo = [c for c in TA_HYDRO_GEO_CANDIDATES if c in df.columns]
ta_residence_soil = [c for c in TA_RESIDENCE_SOIL_CANDIDATES if c in df.columns]

for fs_name, fs in [
    ('SPECTRAL_CORE', spectral_core),
    ('SPECTRAL_PLUS_COORDS', spectral_plus_coords),
    ('MIXED_LITE', mixed_lite),
    ('TA_HYDRO_GEO', ta_hydro_geo),
    ('TA_RESIDENCE_SOIL', ta_residence_soil),
]:
    if len(fs) == 0:
        raise RuntimeError(f'{fs_name} is empty. Check available columns.')

FEATURE_SETS = {
    'SPECTRAL_CORE': spectral_core,
    'SPECTRAL_PLUS_COORDS': spectral_plus_coords,
    'MIXED_LITE': mixed_lite,
    'TA_HYDRO_GEO': ta_hydro_geo,
    'TA_RESIDENCE_SOIL': ta_residence_soil,
}

PRIMARY_FEATURE_SET = 'SPECTRAL_PLUS_COORDS'

print('Feature sets ready:')
for k, v in FEATURE_SETS.items():
    print(f'- {k}: {len(v)} features')
print('PRIMARY_FEATURE_SET:', PRIMARY_FEATURE_SET)


Feature sets ready:
- SPECTRAL_CORE: 19 features
- SPECTRAL_PLUS_COORDS: 21 features
- MIXED_LITE: 38 features
- TA_HYDRO_GEO: 37 features
- TA_RESIDENCE_SOIL: 24 features
PRIMARY_FEATURE_SET: SPECTRAL_PLUS_COORDS


## Preprocessing
Numeric-only, robust, and simple.


In [21]:
def get_preprocessor(features_used):
    numeric_features = [c for c in features_used if pd.api.types.is_numeric_dtype(df[c])]
    if not numeric_features:
        raise RuntimeError('No numeric features available for preprocessing.')
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', RobustScaler())
            ]), numeric_features),
        ],
        remainder='drop'
    )
    return preprocessor


## Models and shortlist
Small model family only.


In [22]:
MODEL_BANK = {
    'RF_baseline': RandomForestRegressor(
        n_estimators=600,
        max_depth=16,
        min_samples_leaf=4,
        random_state=42,
        n_jobs=-1,
    ),
    'ET_baseline': ExtraTreesRegressor(
        n_estimators=700,
        max_depth=18,
        min_samples_leaf=3,
        random_state=42,
        n_jobs=-1,
    ),
    'HGB_baseline': HistGradientBoostingRegressor(
        max_iter=450,
        learning_rate=0.05,
        max_leaf_nodes=31,
        min_samples_leaf=25,
        l2_regularization=0.05,
        random_state=42,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=25,
    ),
}

TARGET_SWEEP = {
    'Total Alkalinity': [
        ('SPECTRAL_PLUS_COORDS', 'RF_baseline'),
        ('TA_HYDRO_GEO', 'RF_baseline'),
        ('TA_HYDRO_GEO', 'ET_baseline'),
        ('TA_RESIDENCE_SOIL', 'RF_baseline'),
        ('TA_HYDRO_GEO', 'HGB_baseline'),
    ],
    'Electrical Conductance': [
        ('SPECTRAL_CORE', 'RF_baseline'),
        ('SPECTRAL_PLUS_COORDS', 'ET_baseline'),
        ('MIXED_LITE', 'ET_baseline'),
        ('SPECTRAL_PLUS_COORDS', 'RF_baseline'),
    ],
    'Dissolved Reactive Phosphorus': [
        ('SPECTRAL_CORE', 'RF_baseline'),
        ('SPECTRAL_PLUS_COORDS', 'RF_baseline'),
        ('SPECTRAL_PLUS_COORDS', 'ET_baseline'),
    ],
}
print('Simple model sweep prepared (TA now uses TA-specific feature combinations).')


Simple model sweep prepared (TA now uses TA-specific feature combinations).


## Grouped evaluation helpers


In [23]:
def target_key(target_name):
    return target_name.replace(' ', '')
def compute_selection_score(overall_r2, holdout_r2, min_fold_r2):
    hold = 0.0 if pd.isna(holdout_r2) else float(holdout_r2)
    worst = 0.0 if pd.isna(min_fold_r2) else float(min_fold_r2)
    return 0.55 * float(overall_r2) + 0.30 * hold + 0.15 * worst
def grouped_oof_eval(df_local, target, estimator, features_used):
    d = df_local.copy()
    X = d[features_used].reset_index(drop=True)
    y = d[target].astype(float).reset_index(drop=True)
    groups = d['spatial_group'].astype(str).reset_index(drop=True)
    n_groups = int(groups.nunique())
    if n_groups < 2:
        raise RuntimeError('Need at least 2 groups for GroupKFold.')
    n_splits = min(int(CV_N_SPLITS), n_groups)
    gkf = GroupKFold(n_splits=n_splits)
    pred = np.full(len(d), np.nan, dtype=float)
    dummy_pred = np.full(len(d), np.nan, dtype=float)
    fold_rows = []
    for fold_id, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups=groups), start=1):
        pipe = Pipeline([
            ('preprocessor', get_preprocessor(features_used)),
            ('model', clone(estimator))
        ])
        X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
        y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]
        pipe.fit(X_tr, y_tr)
        p = np.asarray(pipe.predict(X_te), dtype=float)
        pred[test_idx] = p
        dm = float(np.median(y_tr))
        dmp = np.full(len(test_idx), dm, dtype=float)
        dummy_pred[test_idx] = dmp
        fold_rows.append({
            'fold': fold_id,
            'n': int(len(test_idx)),
            'r2': float(r2_score(y_te, p)),
            'rmse': float(np.sqrt(mean_squared_error(y_te, p))),
            'mae': float(mean_absolute_error(y_te, p)),
        })
    if np.isnan(pred).any() or np.isnan(dummy_pred).any():
        raise RuntimeError('NaN in OOF predictions.')
    fold_df = pd.DataFrame(fold_rows)
    holdout_r2 = np.nan
    dummy_holdout_r2 = np.nan
    hold_mask = d['is_pseudo_valid'].astype(bool).reset_index(drop=True)
    if hold_mask.any() and int((~hold_mask).sum()) >= 2 and int(hold_mask.sum()) >= 2:
        hold_pipe = Pipeline([
            ('preprocessor', get_preprocessor(features_used)),
            ('model', clone(estimator))
        ])
        hold_pipe.fit(X.loc[~hold_mask], y.loc[~hold_mask])
        hold_pred = np.asarray(hold_pipe.predict(X.loc[hold_mask]), dtype=float)
        holdout_r2 = float(r2_score(y.loc[hold_mask], hold_pred))
        dummy_hold = float(np.median(y.loc[~hold_mask]))
        dummy_hold_pred = np.full(int(hold_mask.sum()), dummy_hold, dtype=float)
        dummy_holdout_r2 = float(r2_score(y.loc[hold_mask], dummy_hold_pred))
    return {
        'pred': pred,
        'r2': float(r2_score(y, pred)),
        'mean_fold_r2': float(fold_df['r2'].mean()),
        'min_fold_r2': float(fold_df['r2'].min()),
        'holdout_r2': holdout_r2,
        'rmse': float(np.sqrt(mean_squared_error(y, pred))),
        'mae': float(mean_absolute_error(y, pred)),
        'dummy_r2': float(r2_score(y, dummy_pred)),
        'dummy_holdout_r2': dummy_holdout_r2,
        'delta_r2_vs_dummy': float(r2_score(y, pred) - r2_score(y, dummy_pred)),
        'delta_holdout_r2_vs_dummy': np.nan if pd.isna(holdout_r2) or pd.isna(dummy_holdout_r2) else float(holdout_r2 - dummy_holdout_r2),
        'fold_df': fold_df,
    }
def fit_full_and_save(df_local, target, estimator, features_used, run_name):
    X = df_local[features_used]
    y = df_local[target].astype(float)
    pre = get_preprocessor(features_used)
    Xp = pre.fit_transform(X)
    mdl = clone(estimator)
    mdl.fit(Xp, y)
    preproc_path = os.path.join(ARTIFACT_DIR, f'{run_name}__preproc.joblib')
    model_path = os.path.join(ARTIFACT_DIR, f'{run_name}__model.joblib')
    joblib.dump(pre, preproc_path)
    joblib.dump(mdl, model_path)
    return preproc_path, model_path


## Stage 1: Scout run


In [24]:
rows_scout = []
for target in tqdm(TARGET_COLS, total=len(TARGET_COLS), desc='Scout targets'):
    recipes = TARGET_SWEEP[target]
    for feature_set_name, model_name in tqdm(recipes, total=len(recipes), desc=f'Scout {target}', leave=False):
        features = FEATURE_SETS[feature_set_name]
        estimator = clone(MODEL_BANK[model_name])
        out = grouped_oof_eval(
            df_local=df,
            target=target,
            estimator=estimator,
            features_used=features,
        )
        selection_score = compute_selection_score(
            overall_r2=out['r2'],
            holdout_r2=out['holdout_r2'],
            min_fold_r2=out['min_fold_r2'],
        )
        rows_scout.append({
            'target': target,
            'feature_set': feature_set_name,
            'model_name': model_name,
            'features_used_json': json.dumps(features),
            'r2': out['r2'],
            'mean_fold_r2': out['mean_fold_r2'],
            'min_fold_r2': out['min_fold_r2'],
            'holdout_r2': out['holdout_r2'],
            'dummy_r2': out['dummy_r2'],
            'dummy_holdout_r2': out['dummy_holdout_r2'],
            'delta_r2_vs_dummy': out['delta_r2_vs_dummy'],
            'delta_holdout_r2_vs_dummy': out['delta_holdout_r2_vs_dummy'],
            'selection_score': selection_score,
            'rmse': out['rmse'],
            'mae': out['mae'],
        })
scout_df = pd.DataFrame(rows_scout).sort_values(
    ['target', 'selection_score', 'r2', 'min_fold_r2'],
    ascending=[True, False, False, False]
).reset_index(drop=True)
print('Scout results:')
display(scout_df)


Scout targets:   0%|          | 0/3 [00:00<?, ?it/s]

Scout Total Alkalinity:   0%|          | 0/5 [00:00<?, ?it/s]

Scout Electrical Conductance:   0%|          | 0/4 [00:00<?, ?it/s]

Scout Dissolved Reactive Phosphorus:   0%|          | 0/3 [00:00<?, ?it/s]

Scout results:


,target,feature_set,model_name,features_used_json,r2,mean_fold_r2,min_fold_r2,holdout_r2,dummy_r2,dummy_holdout_r2,delta_r2_vs_dummy,delta_holdout_r2_vs_dummy,selection_score,rmse,mae
0,Dissolved Reactive Phosphorus,SPECTRAL_PLUS_COORDS,ET_baseline,"[""nir"", ""green"", ""swir16"", ""swir22"", ""NDMI"", ""...",0.052228,-0.182451,-0.503929,0.048316,-0.212969,-0.056404,0.265196,0.104720,-0.032369,49.628386,34.686191
1,Dissolved Reactive Phosphorus,SPECTRAL_PLUS_COORDS,RF_baseline,"[""nir"", ""green"", ""swir16"", ""swir22"", ""NDMI"", ""...",-0.114327,-0.489094,-1.081316,0.016321,-0.212969,-0.056404,0.098641,0.072725,-0.220181,53.812668,36.196358
2,Dissolved Reactive Phosphorus,SPECTRAL_CORE,RF_baseline,"[""nir"", ""green"", ""swir16"", ""swir22"", ""NDMI"", ""...",-0.120774,-0.470818,-1.219104,-0.305540,-0.212969,-0.056404,0.092195,-0.249136,-0.340953,53.968099,35.271205
3,Electrical Conductance,MIXED_LITE,ET_baseline,"[""nir"", ""green"", ""swir16"", ""swir22"", ""NDMI"", ""...",0.302881,0.149693,-0.071267,0.149174,-0.192364,-1.074325,0.495245,1.223499,0.200647,285.480976,209.915273
4,Electrical Conductance,SPECTRAL_PLUS_COORDS,ET_baseline,"[""nir"", ""green"", ""swir16"", ""swir22"", ""NDMI"", ""...",0.006436,-0.212160,-0.986191,-0.500395,-0.192364,-1.074325,0.198800,0.573930,-0.294507,340.817245,260.982814
5,Electrical Conductance,SPECTRAL_PLUS_COORDS,RF_baseline,"[""nir"", ""green"", ""swir16"", ""swir22"", ""NDMI"", ""...",-0.321501,-0.855420,-2.141713,-1.204999,-0.192364,-1.074325,-0.129137,-0.130674,-0.859582,393.058742,298.770766
6,Electrical Conductance,SPECTRAL_CORE,RF_baseline,"[""nir"", ""green"", ""swir16"", ""swir22"", ""NDMI"", ""...",-0.398069,-0.748097,-2.138807,-1.520039,-0.192364,-1.074325,-0.205706,-0.445714,-0.995771,404.285445,310.342715
7,Total Alkalinity,TA_HYDRO_GEO,ET_baseline,"[""Latitude"", ""Longitude"", ""swir22"", ""NDMI"", ""M...",0.354463,0.130612,-0.161194,0.279944,-0.180880,-0.023319,0.535343,0.303263,0.254759,60.008763,47.061643
8,Total Alkalinity,TA_RESIDENCE_SOIL,RF_baseline,"[""Latitude"", ""Longitude"", ""dem_slope_1km"", ""ba...",0.219518,-0.027489,-0.254862,0.104541,-0.180880,-0.023319,0.400398,0.127859,0.113868,65.983530,52.060172
9,Total Alkalinity,TA_HYDRO_GEO,HGB_baseline,"[""Latitude"", ""Longitude"", ""swir22"", ""NDMI"", ""M...",0.268624,0.037133,-0.308074,0.010362,-0.180880,-0.023319,0.449503,0.033680,0.104640,63.874081,50.188103


## Stage 2: Full finalists


In [25]:
finalist_df = (
    scout_df.sort_values(['target', 'selection_score', 'r2'], ascending=[True, False, False])
    .groupby('target', as_index=False)
    .head(1)
    .reset_index(drop=True)
)
rows_full = []
for _, row in tqdm(finalist_df.iterrows(), total=len(finalist_df), desc='Full finalists'):
    target = row['target']
    model_name = row['model_name']
    feature_set_name = row['feature_set']
    features = json.loads(row['features_used_json'])
    estimator = clone(MODEL_BANK[model_name])
    run_name = f'FULL__{model_name}__{feature_set_name}__{target_key(target)}'
    out = grouped_oof_eval(
        df_local=df,
        target=target,
        estimator=estimator,
        features_used=features,
    )
    preproc_path, model_path = fit_full_and_save(
        df_local=df,
        target=target,
        estimator=estimator,
        features_used=features,
        run_name=run_name,
    )
    selection_score = compute_selection_score(out['r2'], out['holdout_r2'], out['min_fold_r2'])
    rows_full.append({
        'run_name': run_name,
        'target': target,
        'model_name': model_name,
        'feature_set': feature_set_name,
        'features_used_json': json.dumps(features),
        'r2': out['r2'],
        'mean_fold_r2': out['mean_fold_r2'],
        'min_fold_r2': out['min_fold_r2'],
        'holdout_r2': out['holdout_r2'],
        'dummy_holdout_r2': out['dummy_holdout_r2'],
        'delta_holdout_r2_vs_dummy': out['delta_holdout_r2_vs_dummy'],
        'selection_score': selection_score,
        'rmse': out['rmse'],
        'mae': out['mae'],
        'preproc_path': preproc_path,
        'model_path': model_path,
    })
full_df = pd.DataFrame(rows_full).sort_values(['selection_score', 'r2'], ascending=[False, False]).reset_index(drop=True)
print('Full finalists:')
display(full_df)


Full finalists:   0%|          | 0/3 [00:00<?, ?it/s]

Full finalists:


,run_name,target,model_name,feature_set,features_used_json,r2,mean_fold_r2,min_fold_r2,holdout_r2,dummy_holdout_r2,delta_holdout_r2_vs_dummy,selection_score,rmse,mae,preproc_path,model_path
0,FULL__ET_baseline__TA_HYDRO_GEO__TotalAlkalinity,Total Alkalinity,ET_baseline,TA_HYDRO_GEO,"[""Latitude"", ""Longitude"", ""swir22"", ""NDMI"", ""M...",0.354463,0.130612,-0.161194,0.279944,-0.023319,0.303263,0.254759,60.008763,47.061643,../models/final_deadline_simple\FULL__ET_basel...,../models/final_deadline_simple\FULL__ET_basel...
1,FULL__ET_baseline__MIXED_LITE__ElectricalCondu...,Electrical Conductance,ET_baseline,MIXED_LITE,"[""nir"", ""green"", ""swir16"", ""swir22"", ""NDMI"", ""...",0.302881,0.149693,-0.071267,0.149174,-1.074325,1.223499,0.200647,285.480976,209.915273,../models/final_deadline_simple\FULL__ET_basel...,../models/final_deadline_simple\FULL__ET_basel...
2,FULL__ET_baseline__SPECTRAL_PLUS_COORDS__Disso...,Dissolved Reactive Phosphorus,ET_baseline,SPECTRAL_PLUS_COORDS,"[""nir"", ""green"", ""swir16"", ""swir22"", ""NDMI"", ""...",0.052228,-0.182451,-0.503929,0.048316,-0.056404,0.104720,-0.032369,49.628386,34.686191,../models/final_deadline_simple\FULL__ET_basel...,../models/final_deadline_simple\FULL__ET_basel...


## Freeze Manifest


In [26]:
manifest_A = {}
for target in TARGET_COLS:
    tdf = full_df[full_df['target'] == target].copy()
    if tdf.empty:
        continue
    best = tdf.sort_values(['selection_score', 'holdout_r2', 'r2'], ascending=[False, False, False]).iloc[0]
    manifest_A[target] = {
        'run_name': str(best['run_name']),
        'model_name': str(best['model_name']),
        'feature_set': str(best['feature_set']),
        'holdout_r2': float(best['holdout_r2']) if pd.notna(best['holdout_r2']) else None,
        'delta_holdout_r2_vs_dummy': float(best['delta_holdout_r2_vs_dummy']) if pd.notna(best['delta_holdout_r2_vs_dummy']) else None,
        'selection_score': float(best['selection_score']),
        'r2': float(best['r2']),
        'preproc_path': str(best['preproc_path']),
        'model_path': str(best['model_path']),
    }
print('=== MANIFEST A ===')
print(json.dumps(manifest_A, indent=2))


=== MANIFEST A ===
{
  "Total Alkalinity": {
    "run_name": "FULL__ET_baseline__TA_HYDRO_GEO__TotalAlkalinity",
    "model_name": "ET_baseline",
    "feature_set": "TA_HYDRO_GEO",
    "holdout_r2": 0.2799440113440238,
    "delta_holdout_r2_vs_dummy": 0.3032625352194388,
    "selection_score": 0.25475901794201666,
    "r2": 0.35446336022370806,
    "preproc_path": "../models/final_deadline_simple\\FULL__ET_baseline__TA_HYDRO_GEO__TotalAlkalinity__preproc.joblib",
    "model_path": "../models/final_deadline_simple\\FULL__ET_baseline__TA_HYDRO_GEO__TotalAlkalinity__model.joblib"
  },
  "Electrical Conductance": {
    "run_name": "FULL__ET_baseline__MIXED_LITE__ElectricalConductance",
    "model_name": "ET_baseline",
    "feature_set": "MIXED_LITE",
    "holdout_r2": 0.14917437207449635,
    "delta_holdout_r2_vs_dummy": 1.2234993092584356,
    "selection_score": 0.20064690548982872,
    "r2": 0.30288105167144996,
    "preproc_path": "../models/final_deadline_simple\\FULL__ET_baseline__MIX

## Submission helper


In [ ]:
def predict_with_artifact(preproc_path, model_path, X_df):
    pre = joblib.load(preproc_path)
    mdl = joblib.load(model_path)
    Xp = pre.transform(X_df)
    pred = np.asarray(mdl.predict(Xp), dtype=float)
    return np.clip(pred, 0.0, None)
def target_train_median(df_local, target):
    return float(np.nanmedian(df_local[target].astype(float).values))


## Build Shot A / B
- Shot A: direct manifest predictions
- Shot B: DRP conservative blend (small median hedge)


In [ ]:
stamp = time.strftime('%Y%m%d_%H%M')
tpl = pd.read_csv('../data/raw/submission_template.csv')
subA = tpl.copy()
subB = tpl.copy()
for target in TARGET_COLS:
    info = manifest_A[target]
    fs_name = info['feature_set']
    features = FEATURE_SETS[fs_name]
    for f in features:
        if f not in df_test.columns:
            raise RuntimeError(f'Missing test feature for {target}: {f}')
    pred = predict_with_artifact(info['preproc_path'], info['model_path'], df_test[features])
    subA[target] = pred
    subB[target] = pred
# DRP conservative hedge for Shot B
alpha = 0.15
drp_med = target_train_median(df, 'Dissolved Reactive Phosphorus')
subB['Dissolved Reactive Phosphorus'] = (
    (1.0 - alpha) * subB['Dissolved Reactive Phosphorus'].values + alpha * drp_med
)
pathA = f'../data/submission/submission_simple_{stamp}_A.csv'
pathB = f'../data/submission/submission_simple_{stamp}_B_drp_hedge.csv'
subA.to_csv(pathA, index=False)
subB.to_csv(pathB, index=False)
print('Saved submissions:')
print('A:', pathA)
print('B:', pathB)


## Submission diagnostics


In [ ]:
print('Selected models by target:')
for t in TARGET_COLS:
    print(f'- {t}: {manifest_A[t]["model_name"]} | fs={manifest_A[t]["feature_set"]} | holdout_r2={manifest_A[t]["holdout_r2"]}')
print('\nCeiling reference (spatial local expectation):')
for t, (lo, hi) in REALISTIC_LOCAL_CEILING.items():
    print(f'- {t}: ~{lo:.2f} to {hi:.2f}')
